# Word Parsing: Preserve Headings and Tables

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Word documents contain structural objects. Extract paragraphs, heading hierarchy, and tables with provenance instead of flattening everything into one string.

## 30-Second Summary

This notebook parses a repository-owned proposal with `python-docx`. It compares plain paragraph extraction with typed blocks and verifies that five headings and one table remain identifiable.

## Why This Matters

Flattening a DOCX can detach values from table headers and erase section boundaries. Structure-aware blocks support better chunking, citations, and quality checks.

## Scope

| Covers | Does not cover |
|---|---|
| Paragraph styles, headings, table rows, source metadata | Tracked changes, comments, floating text boxes, images, legacy `.doc` |


## Mental Model

```text
DOCX package -> paragraphs + styles + tables -> typed blocks -> validated documents
```


In [1]:
from pathlib import Path
from docx import Document

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file(): return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
DOCX_PATH = REPO_ROOT / "05-DataIngestParsing/data/word_files/proposal.docx"
document = Document(DOCX_PATH)
len(document.paragraphs), len(document.tables)


(19, 1)

## How It Works

A DOCX file is an Open Packaging Convention archive. `python-docx` exposes paragraphs with style names and tables with rows/cells. We convert those objects into bounded records while keeping the source and ordinal.


## Baseline

The baseline joins non-empty paragraphs. It preserves readable prose but omits table content and does not label headings.


In [2]:
baseline_paragraphs = [paragraph.text.strip() for paragraph in document.paragraphs if paragraph.text.strip()]
baseline_text = "\n".join(baseline_paragraphs)
{"paragraphs": len(baseline_paragraphs), "characters": len(baseline_text), "preview": baseline_text[:120]}


{'paragraphs': 19,
 'characters': 558,
 'preview': 'Project Proposal: RAG Implementation\nExecutive Summary\nThis proposal outlines the implementation of a Retrieval-Augmente'}

## Technique Implementation

Typed blocks keep each paragraph's style and each table row's header-to-value mapping. This is still an extraction layer; downstream chunking can group blocks under the nearest heading.


In [3]:
source = DOCX_PATH.relative_to(REPO_ROOT).as_posix()
blocks = []
for ordinal, paragraph in enumerate(document.paragraphs):
    text = paragraph.text.strip()
    if text:
        blocks.append({
            "id": f"proposal:p{ordinal}", "source": source, "kind": "paragraph",
            "style": paragraph.style.name, "content": text,
        })
for table_index, table in enumerate(document.tables):
    headers = [cell.text.strip() for cell in table.rows[0].cells]
    for row_index, row in enumerate(table.rows[1:], start=1):
        values = [cell.text.strip() for cell in row.cells]
        record = dict(zip(headers, values, strict=True))
        blocks.append({
            "id": f"proposal:t{table_index}:r{row_index}", "source": source,
            "kind": "table_row", "record": record,
            "content": "; ".join(f"{key}: {value}" for key, value in record.items()),
        })
[(block["kind"], block.get("style"), block["content"][:65]) for block in blocks]


[('paragraph', 'Title', 'Project Proposal: RAG Implementation'),
 ('paragraph', 'Heading 1', 'Executive Summary'),
 ('paragraph',
  'Normal',
  'This proposal outlines the implementation of a Retrieval-Augmente'),
 ('paragraph', 'Heading 1', 'Objectives'),
 ('paragraph', 'Normal', 'Key objectives include:'),
 ('paragraph', 'List Bullet', '• Improve information retrieval accuracy'),
 ('paragraph', 'List Bullet', '• Reduce response time for customer queries'),
 ('paragraph', 'List Bullet', '• Integrate with existing knowledge base'),
 ('paragraph', 'Heading 1', 'Budget and Timeline'),
 ('paragraph', 'Normal', 'Budget: $50,000'),
 ('paragraph', 'Normal', 'Timeline: 3 months'),
 ('paragraph', 'Normal', 'Team: 4 developers, 1 project manager'),
 ('paragraph', 'Heading 1', 'Technical Requirements'),
 ('paragraph', 'Normal', 'Required technologies:'),
 ('paragraph', 'List Bullet', '- Python 3.8+'),
 ('paragraph', 'List Bullet', '- OpenAI API access'),
 ('paragraph', 'List Bullet', '- Vector d

## Controlled Experiment

We test structural coverage: heading blocks should match the source's Heading styles, and table rows should remain queryable as header-value records. The baseline cannot satisfy either requirement.


In [4]:
heading_blocks = [block for block in blocks if block.get("style", "").startswith("Heading")]
table_blocks = [block for block in blocks if block["kind"] == "table_row"]
experiment_result = {
    "baseline_has_heading_labels": False,
    "baseline_has_table_rows": False,
    "heading_count": len(heading_blocks),
    "table_row_count": len(table_blocks),
    "typed_block_count": len(blocks),
}
experiment_result


{'baseline_has_heading_labels': False,
 'baseline_has_table_rows': False,
 'heading_count': 5,
 'table_row_count': 3,
 'typed_block_count': 22}

## Evaluation

The typed representation preserves **5 heading blocks** and the rows from the document's single table; the paragraph-only baseline preserves neither signal. The check covers this fixture, not every object a DOCX can contain.


In [5]:
assert experiment_result["heading_count"] == 5
assert experiment_result["table_row_count"] > 0
assert all(block["source"] == source for block in blocks)
assert any("Budget" in block["content"] or "budget" in block["content"] for block in blocks)
print(f"Word checks passed for {len(blocks)} typed blocks.")


Word checks passed for 22 typed blocks.


## Decision Guide

| Need | Approach |
|---|---|
| Simple prose | Paragraph extraction with styles |
| Tables | Header-aware row records |
| Exact visual fidelity | Render plus layout-aware extraction |
| Legacy `.doc` | Convert in a controlled pipeline first |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Table values lose meaning | Headers dropped | Map headers to every row |
| Sections merge | Styles ignored | Preserve heading level/style |
| Text is missing | Content lives in shapes/headers | Add object-specific extraction and fixtures |
| Duplicate rows | Merged cells expanded | Detect spans and normalize carefully |


## Production Notes

### Observability
Track paragraph, heading, table, row, and empty-block counts by parser version.

### Safety and Guardrails
Treat macros, links, and embedded objects as untrusted; this lesson reads `.docx` content only.

### Latency and Cost
Local XML parsing is inexpensive; rendering, OCR, and layout models are slower fallbacks.


## Practice

Add a merged-cell table and a Heading 2 subsection. Specify the expected records before changing the parser.

## Recall

Toggle - Recall: Why keep paragraph styles?
They expose section hierarchy without guessing from font size.

Toggle - Recall: Why turn table rows into mappings?
Headers give each value meaning and improve retrieval/citation context.

## Sources

- [python-docx user guide](https://python-docx.readthedocs.io/en/latest/)
- Repository fixture: `proposal.docx`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for paragraphs and simple tables | Add merged cells and non-body object fixtures |
